<div style="
    background-image: linear-gradient(rgba(15, 23, 42, 0.85), rgba(15, 23, 42, 0.85)), url('https://images.unsplash.com/photo-1518770660439-4636190af475?auto=format&fit=crop&w=1200&q=80');
    background-size: cover;
    background-position: center;
    padding: 32px;
    border-radius: 12px;
    color: #F8FAFC;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.3);
">
    <h1 style="color: #FFFFFF; border-bottom: 2px solid #38BDF8; padding-bottom: 12px; margin-top: 0; font-size: 2.2em; font-weight: 700;">
        Amazon ML Challenge 2026 — Business Entity Resolution
    </h1>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E2E8F0; margin-bottom: 24px;">
        A fast, interactive end-to-end pipeline for the Amazon ML Challenge 2026 dataset. This notebook treats the task as a <strong>large-scale entity-resolution problem</strong>: each Source 1 business must be linked to <strong>zero, one, or many</strong> corresponding records in Source 2 and Source 3.
    </p>
    <details style="background: rgba(255, 255, 255, 0.08); border: 1px solid rgba(255, 255, 255, 0.15); padding: 16px; border-radius: 8px; cursor: pointer; transition: all 0.3s ease;">
        <summary style="font-size: 1.15em; font-weight: 600; color: #38BDF8; outline: none;">
            Click to expand/collapse pipeline workflow
        </summary>
        <ul style="margin-top: 16px; margin-bottom: 8px; line-height: 1.8; color: #F1F5F9; padding-left: 20px;">
            <li>Fast dataset discovery and schema inspection</li>
            <li>Interactive EDA without loading unnecessary columns</li>
            <li>Robust text normalization for names and addresses</li>
            <li>Country-aware blocking that keeps the pipeline open-set</li>
            <li>Exact and high-confidence candidate generation</li>
            <li>Precision-oriented matching designed around <strong>F0.5</strong></li>
            <li>A lightweight validation framework using the training ground truth</li>
            <li>Test-set inference</li>
            <li><code>matching_results.tsv</code> and <code>candidate_pairs.tsv</code> output generation</li>
            <li>Submission-format validation</li>
        </ul>
    </details>
    <div style="margin-top: 24px; padding: 16px; background: rgba(56, 189, 248, 0.1); border-left: 4px solid #38BDF8; border-radius: 4px;">
        <strong style="color: #38BDF8;">Design Principle:</strong> 
        The dataset is very large, so this notebook deliberately avoids expensive all-pairs fuzzy matching. Instead, it uses <strong>blocking + deterministic/high-confidence matching</strong>, which is significantly faster and easier to audit.
    </div>

### What's new in this version

Every original section is still here; upgrades are added as extra cells marked **"Upgrade"**
(plus three small in-place edits, with the replaced lines kept as comments).

| # | Problem in the previous version | Fix |
|---|---|---|
| 1 | `[^a-z0-9\s]` deleted every non-English character: Devanagari/Tamil names became **empty strings**, "Société" became "soci t" | Unicode-safe normalization (8b) |
| 2 | Empty name/address produced keys like `"India\|"`, shared by *every* empty-field record in the country, so each became a candidate for all the others | Empty fields never form a key (14b) |
| 3 | `FAST_DEV_MODE = True` only predicted the first 25k test S1, so the submission would miss almost every entity | Full test set by default (24) |
| 4 | "Block D" was described but never built: blocking was exact-name / exact-address only | 4 extra keys: token-order-free name, postal+name prefix, postal+street number (DBA / trade names at the same address), street number+name prefix (14b) |
| 5 | Section 23 tuned thresholds, but the final matcher still used the hand-set ones | Best configuration is carried into the final matching (23d) |
| 6 | Validation loaded all ~10M S2/S3 rows into memory (plus a dict per row) to score 2,000 entities | Streaming loader keeps only records that share a blocking key with some S1: lossless for blocking (14b, 18, 24) |
| 7 | Hand-set rules only | A LightGBM pairwise matcher trained on ground-truth-labelled candidates, compared against the tuned rules on held-out entities; the winner is used (23b-23d, 27b) |
| 8 | No measure of how many true matches blocking even finds | Blocking-recall + "perfect matcher" ceiling report (21b) |
| 9 | Official validator not run | Runs `validate_submission.py` if it's attached (29b) |

## 1.Environment setup

In [ ]:
# ============================================================
# 1. SETUP — FAST, LIGHTWEIGHT, KAGGLE-FRIENDLY
# ============================================================
import os
import re
import gc
import time
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Optional high-performance dataframe engine.
try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

# Fast string similarity library. Kaggle commonly has it installed.
try:
    from rapidfuzz import fuzz
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False

print("Environment ready")
print("Polars:", HAS_POLARS)
print("RapidFuzz:", HAS_RAPIDFUZZ)

## 2. Locate the dataset automatically

Kaggle mounts datasets under `/kaggle/input/<dataset-slug>/`.

We will **search rather than hard-code a single versioned path**, so the notebook remains reusable if the dataset version changes.

The official challenge format uses TSV files. In particular, the ground-truth file contains a Source 1 ID and a comma-separated list of matching Source 2/3 IDs.

In [ ]:
import os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

# Print all available datasets in /kaggle/input
print("Available datasets in /kaggle/input:")
for item in INPUT_ROOT.iterdir():
    print(" -", item.name)

# Attempt fuzzy match for Amazon datasets or grab the first directory available
dataset_dirs = [p for p in INPUT_ROOT.iterdir() if p.is_dir()]

# Upgrade: prefer whichever attached dataset actually contains the challenge files, so an
# unrelated dataset attached alongside can't be picked by mistake.
_hits = sorted(INPUT_ROOT.rglob("train_source1.tsv"))
DATA_ROOT = (INPUT_ROOT / _hits[0].relative_to(INPUT_ROOT).parts[0]) if _hits else None

# Try to find a directory containing 'amazon' or fallback to the first attached dataset
if DATA_ROOT is None:
    DATA_ROOT = next((p for p in dataset_dirs if "amazon" in p.name.lower()), None)

if DATA_ROOT is None and dataset_dirs:
    DATA_ROOT = dataset_dirs[0]

if DATA_ROOT is None:
    raise FileNotFoundError(
        "No datasets found in /kaggle/input. Ensure you have added the dataset in the right-side panel."
    )

print("\nSelected DATA_ROOT:", DATA_ROOT)

# List all files inside the directory
all_files = sorted(DATA_ROOT.rglob("*"))
for f in all_files:
    if f.is_file():
        print(" -", f.relative_to(DATA_ROOT))

## 3. Confirm the challenge file structure

A crucial detail is that these are **tab-separated files**. Reading them as comma-separated CSVs can collapse the entire row into one column.

In [ ]:
# ============================================================
# 3. RESOLVE TRAIN / TEST PATHS
# ============================================================
def find_file(filename):
    hits = list(DATA_ROOT.rglob(filename))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename}")
    return hits[0]

FILES = {
    "train_s1": find_file("train_source1.tsv"),
    "train_s2": find_file("train_source2.tsv"),
    "train_s3": find_file("train_source3.tsv"),
    "ground_truth": find_file("train_ground_truth.tsv"),
    "test_s1": find_file("test_source1.tsv"),
    "test_s2": find_file("test_source2.tsv"),
    "test_s3": find_file("test_source3.tsv"),
}

for k, v in FILES.items():
    print(f"{k:14s} -> {v}")

## 4. Dataset size — measure first, optimize second

This challenge is genuinely large. A published dataset card reports roughly **26.4 million records across train and test**, so a conventional pandas `read_csv()` of every file is not the best default. citeturn0search2

We therefore use:

- **Polars** when available
- column projection
- small previews for EDA
- sampling for expensive diagnostics
- vectorized normalization
- blocking instead of Cartesian joins

In [ ]:
# ============================================================
# 4. FILE SIZES + SMALL PREVIEWS
# ============================================================
for k, path in FILES.items():
    size_mb = path.stat().st_size / (1024**2)
    print(f"{k:14s} {size_mb:10.1f} MB")

def preview_tsv(path, n=5):
    return pd.read_csv(path, sep="\t", nrows=n)

print("\nTRAIN SOURCE 1 PREVIEW")
display(preview_tsv(FILES["train_s1"]))

print("\nGROUND TRUTH PREVIEW")
display(preview_tsv(FILES["ground_truth"]))

## 5. Load only what we need

For this challenge, the core fields are:

- `entity_id`
- `business_name`
- `business_address`
- `country`

The ground truth contains:

- `source1_entity_id`
- `matched_entity_ids`

Keeping the schema narrow reduces memory pressure and makes subsequent operations faster.

In [ ]:
# ============================================================
# 5. SCHEMA CHECK
# ============================================================
CORE_COLS = ["entity_id", "business_name", "business_address", "country"]

for name in ["train_s1", "train_s2", "train_s3", "test_s1", "test_s2", "test_s3"]:
    df = pd.read_csv(FILES[name], sep="\t", nrows=3)
    print(f"\n{name}")
    print(df.dtypes)
    print("columns:", list(df.columns))

## 6. Fast exploratory sample

Full-scale EDA on tens of millions of rows is usually unnecessary.

Instead, we take a reproducible sample from each source and inspect:

- missingness
- country distribution
- text lengths
- duplicate normalized values
- example noise patterns

This gives us useful structural information without turning EDA into the slowest part of the notebook.

In [ ]:
# ============================================================
# 6. SAMPLE-BASED EDA
# ============================================================
SAMPLE_N = 50_000
RANDOM_STATE = 42

def sample_tsv(path, n=SAMPLE_N, seed=RANDOM_STATE):
    return pd.read_csv(
        path,
        sep="\t",
        usecols=CORE_COLS,
        nrows=n
    )

eda_s1 = sample_tsv(FILES["train_s1"])
eda_s2 = sample_tsv(FILES["train_s2"])
eda_s3 = sample_tsv(FILES["train_s3"])

for label, df in [("S1", eda_s1), ("S2", eda_s2), ("S3", eda_s3)]:
    print(f"\n===== {label} =====")
    print("rows:", len(df))
    print("missing:")
    display(df.isna().mean().mul(100).round(2).to_frame("missing_%"))
    print("countries:")
    display(df["country"].value_counts(dropna=False).head(15).to_frame("count"))

## 7. Visualize the sample

The following compact dashboard gives us a quick feel for the data.

Because the challenge is entity resolution rather than classical classification, the most useful first plots are:

1. records by country
2. business-name length
3. address length
4. missing-value rates

These are sampled visualizations, not estimates intended to replace the full dataset statistics.

In [ ]:
# ============================================================
# 7. QUICK VISUAL DASHBOARD
# ============================================================
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(14, 8))

ax1 = fig.add_subplot(2, 2, 1)
eda_s1["country"].value_counts().head(10).plot(kind="bar", ax=ax1)
ax1.set_title("S1 Sample — Country Distribution")
ax1.set_ylabel("Records")
ax1.tick_params(axis="x", rotation=45)

ax2 = fig.add_subplot(2, 2, 2)
eda_s1["business_name"].fillna("").str.len().clip(upper=100).plot(kind="hist", bins=40, ax=ax2)
ax2.set_title("Business Name Length")
ax2.set_xlabel("Characters")

ax3 = fig.add_subplot(2, 2, 3)
eda_s1["business_address"].fillna("").str.len().clip(upper=250).plot(kind="hist", bins=40, ax=ax3)
ax3.set_title("Business Address Length")
ax3.set_xlabel("Characters")

ax4 = fig.add_subplot(2, 2, 4)
miss = eda_s1.isna().mean().mul(100)
miss.plot(kind="bar", ax=ax4)
ax4.set_title("Missingness — S1 Sample")
ax4.set_ylabel("%")

plt.tight_layout()
plt.show()

## 8. Build robust text normalization

Business names and addresses can differ because of:

- punctuation
- case
- legal suffixes
- `&` versus `and`
- abbreviations
- whitespace
- Unicode variations
- transliteration/noisy formatting

We therefore maintain **multiple representations** instead of destroying information with one aggressive normalization step.

### Representations

- `norm_name`: conservative normalized name
- `compact_name`: whitespace-free normalized name
- `norm_address`: normalized address
- `compact_address`: compact address
- token signatures for blocking

This is especially useful because a single blocking key can create false negatives.

In [ ]:
# ============================================================
# 8. NORMALIZATION FUNCTIONS
# ============================================================
LEGAL_SUFFIXES = {
    "incorporated", "inc", "corporation", "corp",
    "limited", "ltd", "llc", "llp",
    "private", "pvt", "company", "co",
    "limitedliabilitycompany"
}

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_name(x):
    s = normalize_text(x)
    tokens = [t for t in s.split() if t not in LEGAL_SUFFIXES]
    return " ".join(tokens)

def normalize_address(x):
    return normalize_text(x)

def compact(s):
    return re.sub(r"\s+", "", s)

def token_signature(s):
    return " ".join(sorted(set(s.split())))

for df in [eda_s1]:
    df["norm_name"] = df["business_name"].map(normalize_name)
    df["norm_address"] = df["business_address"].map(normalize_address)
    df["compact_name"] = df["norm_name"].map(compact)
    df["compact_address"] = df["norm_address"].map(compact)

display(
    eda_s1[
        ["business_name", "norm_name", "business_address", "norm_address"]
    ].head(15)
)

## 8b. Upgrade: Unicode-safe normalization (India & France)

`re.sub(r"[^a-z0-9\s]", " ", x)` above keeps only ASCII letters and digits. On this dataset that
means:

- **Devanagari / Tamil business names become empty strings.** Every such record then has no name
  signal at all.
- **French accents split words**: "Société Générale" → "soci t g n rale".

The replacement below keeps the exact same behaviour for plain-ASCII text (the fast path, and
the vast majority of rows), and for everything else:

- strips accents from Latin letters (é → e) so "Société" and "Societe" agree
- keeps letters, digits **and combining marks** of every other script (Devanagari/Tamil vowel
  signs are combining marks; removing them corrupts the word)
- converts non-ASCII digits (e.g. Devanagari ४००००१) to ASCII
- drops placeholder tokens (`null`, `nan`, `none`)

It also expands common address abbreviations (St → street, Rd → road, ...) so exact-address
agreement doesn't depend on how a source abbreviates, and extracts the postal code and the street
number, used by the new blocking keys and features.

Everything later in the notebook calls these functions by name, so redefining them here upgrades
the whole pipeline.

In [ ]:
# ============================================================
# 8b. UPGRADE — UNICODE-SAFE NORMALIZATION
# ============================================================
import unicodedata

_ASCII_PUNCT_RE = re.compile(r"[^a-z0-9\s]")
NULL_TOKENS = {"null", "nan", "none"}


def _clean_unicode(x):
    """Non-ASCII path: drop accents on Latin letters only, keep other scripts' letters, digits
    and combining marks, map every other character to a space."""
    out = []
    prev_latin = False
    for ch in unicodedata.normalize("NFKD", x):
        if unicodedata.combining(ch):
            if not prev_latin:
                out.append(ch)
            continue
        cat = unicodedata.category(ch)
        prev_latin = ch.isascii() and ch.isalpha()
        if cat == "Nd":
            out.append(str(unicodedata.digit(ch, 0)))
        elif cat[0] in "LMN" or ch.isspace():
            out.append(ch)
        else:
            out.append(" ")
    return unicodedata.normalize("NFC", "".join(out))


def normalize_text(x):
    if not isinstance(x, str):
        if x is None or pd.isna(x):
            return ""
        x = str(x)
    x = x.lower().replace("&", " and ")
    x = _ASCII_PUNCT_RE.sub(" ", x) if x.isascii() else _clean_unicode(x)
    return " ".join(t for t in x.split() if t not in NULL_TOKENS)


# More legal forms (applied everywhere) + French ones (applied only to France rows, in
# prepare_small below -- "SA"/"SAS" are ordinary words or brands elsewhere).
LEGAL_SUFFIXES.update({"the", "plc", "lp", "pllc", "gmbh", "pte", "pty"})
FRENCH_LEGAL_SUFFIXES = {"sarl", "sas", "sasu", "sa", "eurl", "sci", "snc", "scp", "selarl"}

ADDRESS_ABBR = {
    "st": "street", "str": "street", "rd": "road", "ave": "avenue", "av": "avenue",
    "blvd": "boulevard", "bd": "boulevard", "ln": "lane", "dr": "drive", "hwy": "highway",
    "pkwy": "parkway", "ct": "court", "pl": "place", "sq": "square", "cir": "circle",
    "ste": "suite", "apt": "apartment", "fl": "floor", "flr": "floor", "bldg": "building",
    "nr": "near", "opp": "opposite", "mkt": "market",
}


def normalize_address(x):
    return " ".join(ADDRESS_ABBR.get(t, t) for t in normalize_text(x).split())


COUNTRY_ALIASES = {"usa": "us", "u s": "us", "u s a": "us", "united states": "us",
                   "united states of america": "us", "ind": "india", "bharat": "india",
                   "fr": "france", "fra": "france"}


def normalize_country(x):
    c = normalize_text(x)
    return COUNTRY_ALIASES.get(c, c)


def extract_postal(norm_address):
    """India PIN (6 digits) first, else a 5-digit US ZIP / French code; the last one wins,
    since postal codes trail an address."""
    digits = [t for t in norm_address.split() if t.isdigit()]
    six = [t for t in digits if len(t) == 6]
    if six:
        return six[-1]
    five = [t for t in digits if len(t) == 5]
    return five[-1] if five else ""


def first_number(norm_address, postal=""):
    """First number that isn't the postal code: usually the street / building number."""
    for t in norm_address.split():
        if t.isdigit() and t != postal:
            return t
    return ""


# Re-run the sample normalization with the new functions and show what changed.
for df in [eda_s1]:
    df["norm_name"] = df["business_name"].map(normalize_name)
    df["norm_address"] = df["business_address"].map(normalize_address)
    df["compact_name"] = df["norm_name"].map(compact)
    df["compact_address"] = df["norm_address"].map(compact)

for raw in ["Société Générale Bâtiment SARL", "Sharma & Sons Pvt. Ltd.", "शर्मा ट्रेडर्स",
            "12 Rue de la Paix, 75001 Paris", "45 M.G. Rd, Mumbai, ४००००१", "null"]:
    print(f"{raw!r:40} name -> {normalize_name(raw)!r:32} address -> {normalize_address(raw)!r}")

non_ascii = eda_s1[~eda_s1["business_name"].fillna("").map(str.isascii)]
print(f"\nNon-ASCII names in this sample: {len(non_ascii):,} "
      f"(the old normalization turned these into empty or broken strings)")
display(non_ascii[["business_name", "norm_name", "business_address", "norm_address"]].head(10))

## 9. Inspect normalization quality

Normalization should simplify harmless formatting differences without erasing useful identity information.

The table below lets us visually inspect transformations before using them for matching.

If a normalization rule looks too aggressive for your data, modify it here — the rest of the notebook will automatically use the updated functions.

In [ ]:
# ============================================================
# 9. NORMALIZATION INSPECTION
# ============================================================
inspection = eda_s1[
    ["business_name", "norm_name", "business_address", "norm_address", "country"]
].copy()

display(inspection.sample(min(25, len(inspection)), random_state=42))

## 10. Understand the matching structure

The Source 1 table is the reference side. Source 2 and Source 3 contain noisy records that may correspond to Source 1.

The important consequence is:

> We do **not** need to compare S1 against S1.

We only need candidate links:

```text
S1 → S2
S1 → S3
```

Country is a powerful first-level partition. The published dataset analysis reports that the provided ground truth contains no cross-country matches, while also warning that **France appears in test but not training**. Therefore, we use country as a blocking field but never hard-code the allowed countries.

In [ ]:
# ============================================================
# 10. GROUND-TRUTH SUMMARY
# ============================================================
gt = pd.read_csv(FILES["ground_truth"], sep="\t", dtype=str).fillna("")

def split_ids(x):
    if not x:
        return []
    return [v.strip() for v in x.split(",") if v.strip()]

gt["match_list"] = gt["matched_entity_ids"].map(split_ids)
gt["n_matches"] = gt["match_list"].str.len()

print("Ground-truth rows:", len(gt))
print("Singleton / no-match S1:", int((gt["n_matches"] == 0).sum()))
print("S1 with >=1 match:", int((gt["n_matches"] > 0).sum()))
print("Maximum matches for one S1:", int(gt["n_matches"].max()))

display(gt["n_matches"].describe().to_frame("match_count"))

## 11. Learn the match-count distribution

The challenge is not strictly one-to-one.

A Source 1 entity can have:

- no matches
- one match
- several matches

Therefore, a correct pipeline must **not** simply choose the single highest-scoring candidate.

We will score candidates independently and then produce a list for every S1 entity.

In [ ]:
# ============================================================
# 11. MATCH COUNT VISUALIZATION
# ============================================================
counts = gt["n_matches"].value_counts().sort_index()

ax = counts.head(20).plot(kind="bar", figsize=(12, 4))
ax.set_title("Ground Truth — Number of Matches per Source 1 Entity")
ax.set_xlabel("Number of matched S2/S3 records")
ax.set_ylabel("Number of S1 entities")
plt.xticks(rotation=0)
plt.show()

## 12. Exact-match baseline

Before adding fuzzy logic, establish a strong and interpretable baseline.

We test several exact signals:

1. normalized name
2. normalized address
3. normalized name + address
4. compact name + compact address

Exact agreement is particularly valuable in a precision-heavy metric because accidental false merges are expensive.

The official evaluation uses macro-averaged **F₀.₅**, which gives more weight to precision than recall.

In [ ]:
# ============================================================
# 12. EXACT MATCH HELPERS
# ============================================================
def prepare_small(df):
    out = df[CORE_COLS].copy()
    out = out.fillna("")
    out["norm_name"] = out["business_name"].map(normalize_name)
    out["norm_address"] = out["business_address"].map(normalize_address)
    out["compact_name"] = out["norm_name"].map(compact)
    out["compact_address"] = out["norm_address"].map(compact)
    out["name_addr_key"] = (
        out["country"].astype(str) + "|" +
        out["compact_name"] + "|" +
        out["compact_address"]
    )
    return out

small_s1 = prepare_small(pd.read_csv(FILES["train_s1"], sep="\t", nrows=20_000, dtype=str))
small_s2 = prepare_small(pd.read_csv(FILES["train_s2"], sep="\t", nrows=50_000, dtype=str))
small_s3 = prepare_small(pd.read_csv(FILES["train_s3"], sep="\t", nrows=50_000, dtype=str))

display(small_s1.head())

## 12b. Upgrade: richer record preparation

Same columns as before, plus:

- `country_norm`: case/alias-normalized country (`US`/`USA`/`United States` agree). The original
  `country` column is untouched.
- `postal`, `addr_num`: postal code and street/building number, for blocking and features.
- French legal forms (SARL, SAS, SA, ...) are removed only for France rows.

Uses list comprehensions instead of `.map()`, which is noticeably faster on millions of rows.

In [ ]:
# ============================================================
# 12b. UPGRADE — RECORD PREPARATION
# ============================================================
def _strip_french_legal(name):
    return " ".join(t for t in name.split() if t not in FRENCH_LEGAL_SUFFIXES)


def prepare_small(df):
    out = df[CORE_COLS].copy().fillna("")
    country_map = {c: normalize_country(c) for c in out["country"].unique()}
    out["country_norm"] = out["country"].map(country_map)
    names = [normalize_name(x) for x in out["business_name"].tolist()]
    is_fr = (out["country_norm"] == "france").tolist()
    names = [_strip_french_legal(n) if fr else n for n, fr in zip(names, is_fr)]
    addrs = [normalize_address(x) for x in out["business_address"].tolist()]
    postal = [extract_postal(a) for a in addrs]
    out["norm_name"] = names
    out["norm_address"] = addrs
    out["compact_name"] = [n.replace(" ", "") for n in names]
    out["compact_address"] = [a.replace(" ", "") for a in addrs]
    out["postal"] = postal
    out["addr_num"] = [first_number(a, p) for a, p in zip(addrs, postal)]
    return out


small_s1 = prepare_small(small_s1)
small_s2 = prepare_small(small_s2)
small_s3 = prepare_small(small_s3)
display(small_s1[["business_name", "norm_name", "country_norm", "postal", "addr_num"]].head())

## 13. Blocking strategy

A naive comparison would have approximately:

```text
|S1| × (|S2| + |S3|)
```

candidate pairs — far too many.

Instead we generate candidates through several cheap indexes:

### Block A — exact normalized name
`country + compact_name`

### Block B — exact normalized address
`country + compact_address`

### Block C — exact name + address
`country + compact_name + compact_address`

### Block D — token-prefix key
A compact prefix of the strongest name tokens.

The union of these blocks forms the candidate set. The final matcher then applies stricter rules.

In [ ]:
# ============================================================
# 13. BLOCKING INDEX
# ============================================================
def build_index(df, column):
    idx = defaultdict(list)
    for row in df.itertuples(index=False):
        key = getattr(row, column)
        if key:
            idx[key].append(row.entity_id)
    return idx

def add_block_keys(df):
    df = df.copy()
    df["country_name_key"] = (
        df["country"].astype(str) + "|" + df["compact_name"]
    )
    df["country_addr_key"] = (
        df["country"].astype(str) + "|" + df["compact_address"]
    )
    df["country_name_addr_key"] = (
        df["country"].astype(str) + "|" +
        df["compact_name"] + "|" + df["compact_address"]
    )
    return df

small_s1 = add_block_keys(small_s1)
small_s2 = add_block_keys(small_s2)
small_s3 = add_block_keys(small_s3)

s2_name_idx = build_index(small_s2, "country_name_key")
s2_addr_idx = build_index(small_s2, "country_addr_key")
s3_name_idx = build_index(small_s3, "country_name_key")
s3_addr_idx = build_index(small_s3, "country_addr_key")

print("S2 name index keys:", len(s2_name_idx))
print("S3 name index keys:", len(s3_name_idx))

## 14. Candidate generation for one entity

For each S1 record, we collect the union of candidates returned by the blocking rules.

This is intentionally transparent: if a true match never enters the candidate set, no later model can recover it.

That is why candidate generation should be treated as a first-class part of entity resolution rather than a preprocessing detail.

In [ ]:
# ============================================================
# 14. CANDIDATE GENERATION
# ============================================================
def candidates_for_row(row, indexes):
    candidates = set()

    for key_col, idx in [
        ("country_name_key", indexes["name"]),
        ("country_addr_key", indexes["addr"]),
        ("country_name_addr_key", indexes["name_addr"]),
    ]:
        key = getattr(row, key_col)
        if key:
            candidates.update(idx.get(key, []))

    return candidates

s2_name_addr_idx = build_index(small_s2, "country_name_addr_key")
s3_name_addr_idx = build_index(small_s3, "country_name_addr_key")

INDEXES_S2 = {
    "name": s2_name_idx,
    "addr": s2_addr_idx,
    "name_addr": s2_name_addr_idx,
}
INDEXES_S3 = {
    "name": s3_name_idx,
    "addr": s3_addr_idx,
    "name_addr": s3_name_addr_idx,
}

example = small_s1.iloc[0]
print("Example S1:", example["entity_id"])
print("S2 candidates:", candidates_for_row(example, INDEXES_S2))
print("S3 candidates:", candidates_for_row(example, INDEXES_S3))

## 14b. Upgrade: safer, higher-recall blocking

**1. Empty fields no longer form a key.** Before, a record with an empty name got the key
`"US|"`, shared with *every* other empty-name US record, so each became a candidate for all
the others. With Devanagari names normalized to empty strings (Section 8b), that was a big group.

**2. Four new keys** (Block D from Section 13, made concrete). Each catches matches the exact
keys miss:

| key | catches |
|---|---|
| `country + sorted name tokens` | word-order changes ("Traders Sharma" vs "Sharma Traders") |
| `country + postal + first 4 chars of name` | a typo later in the name, or any address rewrite |
| `country + postal + street number` | a completely different (trade / DBA) name at the same address |
| `country + street number + first 4 chars of name` | as above when one side has no postal code |

**3. Block-size cap.** A key shared by more than `MAX_BLOCK_SIZE` records (a chain's name, a
generic address) is skipped for that index. Those blocks add thousands of candidates per S1 and
almost never contain a true match the narrower keys miss. Exact name+address is never capped.

**4. Keys are stored as 64-bit hashes** (8 bytes instead of ~100-byte strings). This matters at
~26M records.

**5. `load_core_pruned`** streams S2/S3 in chunks and keeps only the records that share at least
one key with some S1. Candidates can *only* come from a shared key, so this drops nothing
blocking could have found. It only saves memory.

In [ ]:
# ============================================================
# 14b. UPGRADE — BLOCKING KEYS, CAPS, PRUNED LOADING
# ============================================================
INDEX_KEY_COLS = {
    "name": "country_name_key",
    "addr": "country_addr_key",
    "name_addr": "country_name_addr_key",
    "sig": "country_sig_key",
    "pin_name": "country_pin_name_key",
    "pin_num": "country_pin_num_key",
    "num_name": "country_num_name_key",
}
MAX_BLOCK_SIZE = 200
UNCAPPED_INDEXES = {"name_addr"}


def _hash_keys(parts, valid):
    """64-bit hash of the '|'-joined parts; 0 (= no key) wherever a required part is empty."""
    joined = pd.Series(["|".join(p) for p in zip(*parts)], dtype=object)
    h = pd.util.hash_pandas_object(joined, index=False).to_numpy()
    return np.where(valid, h, np.uint64(0)).astype(np.uint64)


def add_block_keys(df):
    df = df.copy()
    c = (df["country_norm"] if "country_norm" in df else df["country"]).tolist()
    n = df["compact_name"].tolist()
    a = df["compact_address"].tolist()
    sig = [" ".join(sorted(set(x.split()))) for x in df["norm_name"].tolist()]
    pin = df["postal"].tolist() if "postal" in df else [""] * len(df)
    num = df["addr_num"].tolist() if "addr_num" in df else [""] * len(df)
    n4 = [x[:4] for x in n]
    has_n = np.array([bool(x) for x in n], dtype=bool)
    has_a = np.array([bool(x) for x in a], dtype=bool)
    has_pin = np.array([bool(x) for x in pin], dtype=bool)
    has_num = np.array([bool(x) for x in num], dtype=bool)
    df["country_name_key"] = _hash_keys([c, n], has_n)
    df["country_addr_key"] = _hash_keys([c, a], has_a)
    df["country_name_addr_key"] = _hash_keys([c, n, a], has_n & has_a)
    df["country_sig_key"] = _hash_keys([c, sig], has_n)
    df["country_pin_name_key"] = _hash_keys([c, pin, n4], has_pin & has_n)
    df["country_pin_num_key"] = _hash_keys([c, pin, num], has_pin & has_num)
    df["country_num_name_key"] = _hash_keys([c, num, n4], has_num & has_n)
    return df


def make_indexes_v2(df, cap=MAX_BLOCK_SIZE):
    out = {}
    ids = df["entity_id"].tolist()
    for name, col in INDEX_KEY_COLS.items():
        idx = defaultdict(list)
        for k, eid in zip(df[col].tolist(), ids):
            if k:
                idx[k].append(eid)
        if name not in UNCAPPED_INDEXES:
            idx = {k: v for k, v in idx.items() if len(v) <= cap}
        out[name] = dict(idx)
    return out


def candidates_for_row(row, indexes):
    """Union of every index's block for this row (works with the original 3-index dicts too)."""
    candidates = set()
    for idx_name, idx in indexes.items():
        key = getattr(row, INDEX_KEY_COLS[idx_name], 0)
        if key:
            candidates.update(idx.get(key, ()))
    return candidates


def s1_key_sets(s1_frames):
    return {col: np.unique(np.concatenate([f[col].to_numpy() for f in s1_frames]))
            for col in INDEX_KEY_COLS.values()}


def load_core_pruned(path, keysets, chunksize=500_000):
    """Stream a source file; keep only records sharing a blocking key with some S1."""
    kept, total, t0 = [], 0, time.time()
    for chunk in pd.read_csv(path, sep="\t", usecols=CORE_COLS, dtype=str, chunksize=chunksize):
        total += len(chunk)
        chunk = add_block_keys(prepare_small(chunk))
        mask = np.zeros(len(chunk), dtype=bool)
        for col, ks in keysets.items():
            v = chunk[col].to_numpy()
            mask |= (v != 0) & np.isin(v, ks)
        kept.append(chunk[mask])
        print(f"\r{Path(path).name}: read {total:,} rows, kept {sum(map(len, kept)):,} "
              f"({time.time() - t0:.0f}s)", end="")
    print()
    return pd.concat(kept, ignore_index=True)


# Rebuild the Section 13/14 demo with the upgraded keys.
small_s1 = add_block_keys(small_s1)
small_s2 = add_block_keys(small_s2)
small_s3 = add_block_keys(small_s3)
INDEXES_S2 = make_indexes_v2(small_s2)
INDEXES_S3 = make_indexes_v2(small_s3)

example = small_s1.iloc[0]
print("Example S1:", example["entity_id"], "|", example["business_name"])
print("S2 candidates:", candidates_for_row(example, INDEXES_S2))
print("S3 candidates:", candidates_for_row(example, INDEXES_S3))
print("\nBlocks per index (S2 sample):")
for name, idx in INDEXES_S2.items():
    print(f"  {name:10s} {len(idx):>9,}")

## 15. Candidate scoring

Candidate generation tells us **who might match**.

Scoring tells us **how strong the evidence is**.

We combine inexpensive signals:

- exact normalized name
- exact normalized address
- compact-name equality
- compact-address equality
- token overlap
- optional RapidFuzz similarity

The final decision is deliberately conservative because the challenge's F₀.₅ metric is precision-heavy.

In [ ]:
# ============================================================
# 15. SCORING FUNCTIONS
# ============================================================
def jaccard_tokens(a, b):
    A, B = set(a.split()), set(b.split())
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def pair_features(a, b):
    name_exact = int(a["compact_name"] != "" and a["compact_name"] == b["compact_name"])
    addr_exact = int(a["compact_address"] != "" and a["compact_address"] == b["compact_address"])
    name_token = jaccard_tokens(a["norm_name"], b["norm_name"])
    addr_token = jaccard_tokens(a["norm_address"], b["norm_address"])

    if HAS_RAPIDFUZZ:
        name_fuzzy = fuzz.token_set_ratio(a["norm_name"], b["norm_name"]) / 100
        addr_fuzzy = fuzz.token_set_ratio(a["norm_address"], b["norm_address"]) / 100
    else:
        name_fuzzy = float(name_exact)
        addr_fuzzy = float(addr_exact)

    return {
        "name_exact": name_exact,
        "addr_exact": addr_exact,
        "name_token": name_token,
        "addr_token": addr_token,
        "name_fuzzy": name_fuzzy,
        "addr_fuzzy": addr_fuzzy,
    }

def match_score(f):
    # Conservative weighting: exact agreement dominates.
    return (
        4.0 * f["name_exact"] +
        4.0 * f["addr_exact"] +
        2.0 * f["name_fuzzy"] +
        2.0 * f["addr_fuzzy"] +
        1.0 * f["name_token"] +
        1.0 * f["addr_token"]
    )

## 16. Inspect individual matches interactively

A good entity-resolution notebook should make errors understandable.

The helper below prints:

- S1 record
- candidate record
- feature values
- total score

This is useful when tuning thresholds or investigating false merges.

In [ ]:
# ============================================================
# 16. INTERACTIVE PAIR INSPECTION
# ============================================================
def inspect_pair(a, b):
    f = pair_features(a, b)
    result = pd.DataFrame([{
        "S1": a["entity_id"],
        "candidate": b["entity_id"],
        "country": a["country"],
        "S1_name": a["business_name"],
        "candidate_name": b["business_name"],
        "S1_address": a["business_address"],
        "candidate_address": b["business_address"],
        **f,
        "score": match_score(f)
    }])
    display(result.T)

if len(small_s1) and len(small_s2):
    inspect_pair(small_s1.iloc[0], small_s2.iloc[0])

## 17. Define the official F₀.₅ evaluation

We reproduce the challenge metric locally.

For each Source 1 entity:

- precision = correct predicted matches / predicted matches
- recall = correct predicted matches / true matches
- F₀.₅ combines them with β = 0.5

Singletons matter: an S1 entity with no true matches should receive an empty prediction.

The challenge computes this per S1 entity and then macro-averages the results.

In [ ]:
# ============================================================
# 17. F0.5 METRIC
# ============================================================
def f05_single(true_ids, pred_ids):
    true_set = set(true_ids)
    pred_set = set(pred_ids)

    if not true_set and not pred_set:
        return 1.0
    if not pred_set:
        return 0.0

    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set) if true_set else 0.0

    if precision == 0 and recall == 0:
        return 0.0

    beta2 = 0.25
    return (1 + beta2) * precision * recall / (beta2 * precision + recall)

def macro_f05(y_true, y_pred):
    return np.mean([
        f05_single(t, p)
        for t, p in zip(y_true, y_pred)
    ])

## 18. Build a validation subset

Full fuzzy validation over millions of records is unnecessary while developing.

We create a reproducible validation slice from the training Source 1 entities and evaluate:

1. strict exact matching
2. conservative multi-signal matching

The goal here is **pipeline diagnosis**, not to claim a leaderboard score from a tiny sample.

In [ ]:
# ============================================================
# 18. VALIDATION SUBSET
# ============================================================
# VAL_S1_N = 2_000  (original)
VAL_S1_N = 10_000        # upgrade: 5x more entities -> a much less noisy F0.5 estimate
ML_TRAIN_S1_N = 60_000   # upgrade: separate S1 sample (disjoint from validation) to train the
                         # pairwise model in Section 23b

train_s1_val = pd.read_csv(
    FILES["train_s1"],
    sep="\t",
    usecols=CORE_COLS,
    dtype=str,
    nrows=VAL_S1_N
).fillna("")
train_s1_val = add_block_keys(prepare_small(train_s1_val))

# Upgrade: random S1 entities NOT in the validation slice, for training the pairwise model.
_all_s1 = pd.read_csv(FILES["train_s1"], sep="\t", usecols=CORE_COLS, dtype=str).fillna("")
_pool = _all_s1[~_all_s1["entity_id"].isin(set(train_s1_val["entity_id"]))]
train_s1_ml = _pool.sample(n=min(ML_TRAIN_S1_N, len(_pool)), random_state=RANDOM_STATE)
train_s1_ml = add_block_keys(prepare_small(train_s1_ml))
del _all_s1, _pool
gc.collect()

# Upgrade: stream S2/S3 and keep only records that share a blocking key with some S1 above.
# Lossless for blocking, and avoids holding ~10M normalized rows + a dict per row in memory.
_keysets = s1_key_sets([train_s1_val, train_s1_ml])
train_s2_val = load_core_pruned(FILES["train_s2"], _keysets)
train_s3_val = load_core_pruned(FILES["train_s3"], _keysets)

# Original version (loads every S2/S3 row):
# train_s2_val = pd.read_csv(FILES["train_s2"], sep="\t", usecols=CORE_COLS, dtype=str).fillna("")
# train_s3_val = pd.read_csv(FILES["train_s3"], sep="\t", usecols=CORE_COLS, dtype=str).fillna("")
# train_s2_val = add_block_keys(prepare_small(train_s2_val))
# train_s3_val = add_block_keys(prepare_small(train_s3_val))

print("Validation S1:", len(train_s1_val))
print("ML-training S1:", len(train_s1_ml))
print("Validation S2:", len(train_s2_val))
print("Validation S3:", len(train_s3_val))
print("(S2/S3 counts are the records reachable through blocking, not the whole file)")

## 19. Create reusable source indexes

The same candidate-generation function can now be used for S2 and S3.

We store records in dictionaries by ID for fast lookup after blocking.

In [ ]:
# ============================================================
# 19. VALIDATION INDEXES
# ============================================================
def make_indexes(df):
    return {
        "name": build_index(df, "country_name_key"),
        "addr": build_index(df, "country_addr_key"),
        "name_addr": build_index(df, "country_name_addr_key"),
    }

val_idx_s2 = make_indexes(train_s2_val)
val_idx_s3 = make_indexes(train_s3_val)

s2_by_id = train_s2_val.set_index("entity_id").to_dict("index")
s3_by_id = train_s3_val.set_index("entity_id").to_dict("index")

### 19b. Upgrade: rebuild the indexes with all 7 keys and the block cap

`make_indexes` is re-pointed at the upgraded version, so the final test indexes (Section 25) use
it too.

In [ ]:
make_indexes = make_indexes_v2
val_idx_s2 = make_indexes(train_s2_val)
val_idx_s3 = make_indexes(train_s3_val)
for name in INDEX_KEY_COLS:
    print(f"{name:10s} S2 blocks {len(val_idx_s2[name]):>9,}   S3 blocks {len(val_idx_s3[name]):>9,}")

## 20. Conservative matcher

Because F₀.₅ is precision-heavy, we use a confidence hierarchy:

### High confidence
- exact normalized name **and** exact normalized address

### Strong partial evidence
- exact normalized name + substantial address similarity
- exact normalized address + substantial name similarity

### Fallback
- high fuzzy agreement on both fields

The threshold is intentionally conservative and should be tuned using the validation score rather than arbitrary leaderboard chasing.

In [ ]:
# ============================================================
# 20. CONSERVATIVE MATCH DECISION
# ============================================================
def is_match(f):
    # Highest-confidence rule
    if f["name_exact"] and f["addr_exact"]:
        return True

    # One exact field + strong support from the other
    if f["name_exact"] and f["addr_fuzzy"] >= 0.90:
        return True

    if f["addr_exact"] and f["name_fuzzy"] >= 0.92:
        return True

    # Both fields independently strong
    if f["name_fuzzy"] >= 0.96 and f["addr_fuzzy"] >= 0.88:
        return True

    return False

def predict_for_s1(row, idx2, idx3, by_id2, by_id3):
    candidates = set()
    candidates.update(candidates_for_row(row, idx2))
    candidates.update(candidates_for_row(row, idx3))

    predictions = []

    for cid in candidates:
        if cid.startswith("S2-"):
            c = by_id2.get(cid)
        else:
            c = by_id3.get(cid)

        if c is None:
            continue

        f = pair_features(row, c)
        if is_match(f):
            predictions.append(cid)

    return sorted(set(predictions))

## 21. Run validation

This cell evaluates the complete mini-pipeline on the selected validation slice.

We report:

- macro F₀.₅
- number of predicted links
- percentage of singleton predictions
- candidate counts

These diagnostics are more informative than looking at one score alone.

In [ ]:
# ============================================================
# 21. VALIDATION RUN
# ============================================================
gt_lookup = dict(zip(gt["source1_entity_id"], gt["match_list"]))

# Only score IDs that are present in this validation sample.
val_ids = set(train_s1_val["entity_id"])
val_gt = {k: v for k, v in gt_lookup.items() if k in val_ids}

predictions = {}
candidate_counts = []

t0 = time.time()

# Change this line in Section 21:
for _, row in train_s1_val.iterrows():
    pred = predict_for_s1(
        row, val_idx_s2, val_idx_s3, s2_by_id, s3_by_id
    )
    predictions[row["entity_id"]] = pred

elapsed = time.time() - t0

eval_ids = list(val_gt.keys())
y_true = [val_gt[k] for k in eval_ids]
y_pred = [predictions.get(k, []) for k in eval_ids]

score = macro_f05(y_true, y_pred)

print(f"Validation entities: {len(eval_ids):,}")
print(f"Validation time: {elapsed:.2f}s")
print(f"Macro F0.5: {score:.5f}")
print(f"Predicted links: {sum(map(len, y_pred)):,}")
print(f"Predicted singletons: {sum(len(x)==0 for x in y_pred):,}")

## 21b. Upgrade: blocking recall, the ceiling for any matcher

A true match that never becomes a candidate can't be predicted by any matcher. This reports:

- **pair recall**: share of true (S1, match) pairs that made it into the candidate set
- **per-entity recall**: the same, averaged per S1 that has matches
- **oracle macro F0.5**: the score a *perfect* matcher would get on these candidates. That's the
  best any threshold or model can do with this blocking.

If the matcher score is far below the oracle, improve matching. If the oracle itself is low,
improve blocking.

In [ ]:
# ============================================================
# 21b. BLOCKING RECALL + ORACLE CEILING
# ============================================================
cand_sets = {
    row.entity_id: candidates_for_row(row, val_idx_s2) | candidates_for_row(row, val_idx_s3)
    for row in train_s1_val.itertuples(index=False)
}
tot_true = tot_hit = 0
ent_recall, oracle_pred = [], []
for sid in eval_ids:
    t, c = set(val_gt[sid]), cand_sets.get(sid, set())
    if t:
        hit = len(t & c)
        tot_true += len(t)
        tot_hit += hit
        ent_recall.append(hit / len(t))
    oracle_pred.append(sorted(t & c))

print(f"Pair recall of blocking:        {tot_hit / max(tot_true, 1):.4f}")
print(f"Per-entity recall (with match): {np.mean(ent_recall) if ent_recall else float('nan'):.4f}")
print(f"Avg candidates per S1:          {np.mean([len(cand_sets.get(s, ())) for s in eval_ids]):.2f}")
print(f"Oracle macro F0.5 (ceiling):    {macro_f05(y_true, oracle_pred):.4f}")
print(f"Current rule matcher F0.5:      {score:.4f}")

## 22. Error analysis

A single metric can hide very different behaviors.

We therefore inspect:

- true positives
- false positives
- false negatives
- singleton decisions
- records with many candidates

This is the section to revisit if the validation score is lower than expected.

In [ ]:
# ============================================================
# 22. ERROR ANALYSIS TABLE
# ============================================================
rows = []

for sid in eval_ids:
    t = set(val_gt[sid])
    p = set(predictions.get(sid, []))

    rows.append({
        "source1_entity_id": sid,
        "true_count": len(t),
        "pred_count": len(p),
        "TP": len(t & p),
        "FP": len(p - t),
        "FN": len(t - p),
        "F0.5": f05_single(t, p)
    })

errors = pd.DataFrame(rows)

display(errors.sort_values(["FP", "FN"], ascending=False).head(25))
print("\nAggregate:")
display(errors[["TP", "FP", "FN", "F0.5"]].sum().to_frame("sum"))

## 23. Threshold sensitivity

Rather than blindly increasing recall, compare several conservative fuzzy thresholds.

This is particularly important here because the challenge explicitly makes precision more valuable than recall.

The best threshold should be selected from your validation evidence and then frozen before generating the final test prediction.

In [ ]:
# ============================================================
# 23. SIMPLE THRESHOLD EXPERIMENT
# ============================================================
def make_rule(name_threshold, addr_threshold):
    def rule(f):
        if f["name_exact"] and f["addr_exact"]:
            return True
        if f["name_exact"] and f["addr_fuzzy"] >= addr_threshold:
            return True
        if f["addr_exact"] and f["name_fuzzy"] >= name_threshold:
            return True
        if f["name_fuzzy"] >= name_threshold and f["addr_fuzzy"] >= addr_threshold:
            return True
        return False
    return rule

threshold_results = []

# Small grid — intentionally cheap.
for nt in [0.92, 0.94, 0.96]:
    for at in [0.84, 0.88, 0.92]:
        rule = make_rule(nt, at)
        preds = {}

        for row in train_s1_val.itertuples(index=False):
            cand = set()
            cand.update(candidates_for_row(row, val_idx_s2))
            cand.update(candidates_for_row(row, val_idx_s3))

            # Convert row namedtuple to dict for pair_features compatibility
            row_dict = row._asdict() if hasattr(row, "_asdict") else row

            out = []
            for cid in cand:
                c = s2_by_id.get(cid) if cid.startswith("S2-") else s3_by_id.get(cid)
                if c is None:
                    continue
                
                # Convert candidate namedtuple/object to dict if needed
                c_dict = c._asdict() if hasattr(c, "_asdict") else c

                if rule(pair_features(row_dict, c_dict)):
                    out.append(cid)
            preds[row.entity_id] = sorted(set(out))

        yp = [preds.get(k, []) for k in eval_ids]
        threshold_results.append({
            "name_threshold": nt,
            "address_threshold": at,
            "macro_F0.5": macro_f05(y_true, yp),
            "predicted_links": sum(map(len, yp))
        })

threshold_df = pd.DataFrame(threshold_results).sort_values(
    "macro_F0.5", ascending=False
)
display(threshold_df)

## 23b. Upgrade: a learned pairwise matcher (LightGBM)

The rules above use hand-picked thresholds on four signals. A gradient-boosted model trained on
the **ground-truth-labelled candidate pairs** can weigh more than 20 signals at once: fuzzy name
and address similarity in several flavours, postal / street-number agreement *and conflict*,
length ratio, and how a candidate ranks against the other candidates of the same S1.

Protocol, so the comparison is honest:

1. **Train** on `ML_TRAIN_S1_N` random S1 entities, disjoint from the validation slice.
2. **Tune** every threshold (model *and* rules) on half **A** of the validation entities.
3. **Compare** on half **B**, which neither was tuned on. Both use exactly the same candidate
   pairs.
4. Whichever wins on B is used for the test set. Its threshold is then re-tuned on all
   validation entities.

The model keeps a candidate when `p >= threshold` and `p >= rel × (best p among this S1's
candidates)`. The second condition is a precision lever that suits F0.5: it drops weak
runner-up candidates when there's a clearly better one.

In [ ]:
# ============================================================
# 23b. VECTORIZED PAIR FEATURES
# ============================================================
if HAS_RAPIDFUZZ:
    from rapidfuzz import process as rf_process
    from rapidfuzz.distance import JaroWinkler
    HAS_CPDIST = hasattr(rf_process, "cpdist")
    _TSR, _RATIO, _PARTIAL = fuzz.token_set_ratio, fuzz.ratio, fuzz.partial_ratio
    _JW = JaroWinkler.normalized_similarity
else:
    import difflib
    HAS_CPDIST = False
    _TSR = _RATIO = _PARTIAL = lambda a, b: 100 * difflib.SequenceMatcher(None, a, b).ratio()
    _JW = lambda a, b: difflib.SequenceMatcher(None, a, b).ratio()

SCORING_COLS = ["norm_name", "norm_address", "compact_name", "compact_address", "postal", "addr_num"]


def _pairwise(xs, ys, scorer, scale):
    if HAS_CPDIST:
        return rf_process.cpdist(xs, ys, scorer=scorer, workers=-1, dtype=np.float32) / scale
    return np.fromiter((scorer(x, y) for x, y in zip(xs, ys)), dtype=np.float32, count=len(xs)) / scale


def _jaccard(xs, ys):
    out = np.empty(len(xs), dtype=np.float32)
    for i, (x, y) in enumerate(zip(xs, ys)):
        A, B = set(x.split()), set(y.split())
        out[i] = len(A & B) / len(A | B) if A and B else 0.0
    return out


def lookup_frame(frames):
    f = pd.concat([x[["entity_id"] + SCORING_COLS] for x in frames], ignore_index=True)
    return f.drop_duplicates("entity_id").set_index("entity_id")


def candidate_pairs_frame(s1_df, idx2, idx3):
    s1s, cs = [], []
    for row in s1_df.itertuples(index=False):
        for c in sorted(candidates_for_row(row, idx2) | candidates_for_row(row, idx3)):
            s1s.append(row.entity_id)
            cs.append(c)
    return pd.DataFrame({"source1_entity_id": s1s, "candidate_entity_id": cs})


def pair_feature_table(pairs, s1_lookup, cand_lookup):
    """One row of features per (S1, candidate) pair. `pairs` must contain every candidate of
    each S1 it mentions (the per-S1 rank/gap features are computed within the frame)."""
    a = s1_lookup.loc[pairs["source1_entity_id"].to_numpy()]
    b = cand_lookup.loc[pairs["candidate_entity_id"].to_numpy()]
    an, bn = a["norm_name"].tolist(), b["norm_name"].tolist()
    aa, ba = a["norm_address"].tolist(), b["norm_address"].tolist()
    acn, bcn = a["compact_name"].to_numpy(), b["compact_name"].to_numpy()
    aca, bca = a["compact_address"].to_numpy(), b["compact_address"].to_numpy()
    ap, bp = a["postal"].to_numpy(), b["postal"].to_numpy()
    anum, bnum = a["addr_num"].to_numpy(), b["addr_num"].to_numpy()
    la = np.fromiter(map(len, acn), dtype=np.float32, count=len(acn))
    lb = np.fromiter(map(len, bcn), dtype=np.float32, count=len(bcn))
    pa4 = np.array([x[:4] for x in acn], dtype=object)
    pb4 = np.array([x[:4] for x in bcn], dtype=object)

    F = pd.DataFrame({
        "name_exact": ((acn != "") & (acn == bcn)).astype(np.float32),
        "addr_exact": ((aca != "") & (aca == bca)).astype(np.float32),
        "name_token_set": _pairwise(an, bn, _TSR, 100),
        "name_ratio": _pairwise(an, bn, _RATIO, 100),
        "name_partial": _pairwise(an, bn, _PARTIAL, 100),
        "name_jw": _pairwise(acn.tolist(), bcn.tolist(), _JW, 1),
        "name_jaccard": _jaccard(an, bn),
        "addr_token_set": _pairwise(aa, ba, _TSR, 100),
        "addr_ratio": _pairwise(aa, ba, _RATIO, 100),
        "addr_partial": _pairwise(aa, ba, _PARTIAL, 100),
        "addr_jaccard": _jaccard(aa, ba),
        "postal_match": ((ap != "") & (ap == bp)).astype(np.float32),
        "postal_conflict": ((ap != "") & (bp != "") & (ap != bp)).astype(np.float32),
        "num_match": ((anum != "") & (anum == bnum)).astype(np.float32),
        "num_conflict": ((anum != "") & (bnum != "") & (anum != bnum)).astype(np.float32),
        "name_prefix4": ((pa4 != "") & (pa4 == pb4)).astype(np.float32),
        "name_len_ratio": np.minimum(la, lb) / np.maximum(np.maximum(la, lb), 1),
        "cand_is_s2": pairs["candidate_entity_id"].str.startswith("S2-").to_numpy().astype(np.float32),
    })
    s1 = pairs["source1_entity_id"].to_numpy()
    combo = F["name_token_set"] + F["addr_token_set"]
    grp = combo.groupby(s1)
    F["cand_count"] = grp.transform("size").astype(np.float32)
    F["combo_rank"] = grp.rank(ascending=False, method="min").astype(np.float32)
    F["combo_gap"] = (grp.transform("max") - combo).astype(np.float32)
    F["min_field_sim"] = np.minimum(F["name_token_set"], F["addr_token_set"])
    return F

In [ ]:
# ============================================================
# 23c. TRAIN THE PAIRWISE MODEL
# ============================================================
train_s1_lookup = lookup_frame([train_s1_val, train_s1_ml])
train_cand_lookup = lookup_frame([train_s2_val, train_s3_val])


def label_pairs(pairs):
    pos = {f"{s}|{m}" for s in pairs["source1_entity_id"].unique() for m in gt_lookup.get(s, [])}
    keys = pairs["source1_entity_id"] + "|" + pairs["candidate_entity_id"]
    return keys.isin(pos).to_numpy().astype(np.int8)


t0 = time.time()
pairs_ml = candidate_pairs_frame(train_s1_ml, val_idx_s2, val_idx_s3)
X_ml = pair_feature_table(pairs_ml, train_s1_lookup, train_cand_lookup)
y_ml = label_pairs(pairs_ml)
if len(pairs_ml) == 0 or y_ml.sum() == 0:
    raise ValueError("No labelled training pairs -- check that ML_TRAIN_S1_N > 0 and that blocking "
                     "finds candidates (Section 21b).")
print(f"Training pairs: {len(pairs_ml):,} ({int(y_ml.sum()):,} true matches) "
      f"from {pairs_ml['source1_entity_id'].nunique():,} S1 | features in {time.time() - t0:.1f}s")

# Early-stopping split by whole S1 entity, so no entity's pairs are on both sides.
_ml_ids = pairs_ml["source1_entity_id"].unique()
_es_ids = set(np.random.RandomState(RANDOM_STATE).choice(
    _ml_ids, size=max(1, int(0.15 * len(_ml_ids))), replace=False))
_es = pairs_ml["source1_entity_id"].isin(_es_ids).to_numpy()

try:
    import lightgbm as lgb
    _params = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 63,
               "min_data_in_leaf": 40, "feature_fraction": 0.9, "bagging_fraction": 0.8,
               "bagging_freq": 1, "lambda_l2": 1.0, "verbose": -1, "seed": RANDOM_STATE}
    _dtr = lgb.Dataset(X_ml[~_es], label=y_ml[~_es])
    _des = lgb.Dataset(X_ml[_es], label=y_ml[_es], reference=_dtr)
    pair_model = lgb.train(_params, _dtr, num_boost_round=3000, valid_sets=[_des],
                           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    predict_pairs = lambda X: pair_model.predict(X, num_iteration=pair_model.best_iteration)
    print(f"LightGBM: best iteration {pair_model.best_iteration}")
    importance = pd.Series(pair_model.feature_importance("gain"), index=X_ml.columns)
    display((importance / importance.sum()).sort_values(ascending=False).round(4).to_frame("gain share"))
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    pair_model = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05,
                                                early_stopping=True, random_state=RANDOM_STATE)
    pair_model.fit(X_ml, y_ml)
    predict_pairs = lambda X: pair_model.predict_proba(X)[:, 1]
    print("lightgbm not installed -- using sklearn HistGradientBoostingClassifier")

In [ ]:
# ============================================================
# 23d. HONEST COMPARISON: RULES vs MODEL (held-out validation entities)
# ============================================================
pairs_val = candidate_pairs_frame(train_s1_val, val_idx_s2, val_idx_s3)
X_val = pair_feature_table(pairs_val, train_s1_lookup, train_cand_lookup)
p_val = predict_pairs(X_val)

_perm = np.random.RandomState(RANDOM_STATE + 1).permutation(len(eval_ids))
_ids = np.array(eval_ids, dtype=object)
half_A, half_B = set(_ids[_perm[: len(_perm) // 2]]), set(_ids[_perm[len(_perm) // 2:]])
all_val = set(eval_ids)

_s1_arr = pairs_val["source1_entity_id"].to_numpy()
_c_arr = pairs_val["candidate_entity_id"].to_numpy()


def f05_for_mask(keep, ids):
    pred = {i: [] for i in ids}
    for s, c in zip(_s1_arr[keep], _c_arr[keep]):
        if s in pred:
            pred[s].append(c)
    ids = list(ids)
    return macro_f05([val_gt[i] for i in ids], [pred[i] for i in ids])


_pmax = pd.Series(p_val).groupby(_s1_arr).transform("max").to_numpy()
_ne, _ae = X_val["name_exact"].to_numpy() > 0, X_val["addr_exact"].to_numpy() > 0
_nf, _af = X_val["name_token_set"].to_numpy(), X_val["addr_token_set"].to_numpy()


def ml_mask(t, rel):
    return (p_val >= t) & (p_val >= rel * _pmax)


def rule_mask(nt, at):  # identical logic to make_rule() in Section 23
    return (_ne & _ae) | (_ne & (_af >= at)) | (_ae & (_nf >= nt)) | ((_nf >= nt) & (_af >= at))


_default_rule = (_ne & _ae) | (_ne & (_af >= 0.90)) | (_ae & (_nf >= 0.92)) | ((_nf >= 0.96) & (_af >= 0.88))
ML_GRID = [(round(float(t), 3), rel) for t in np.arange(0.05, 0.96, 0.025) for rel in (0.0, 0.5, 0.8, 0.95)]
RULE_GRID = [(nt, at) for nt in (0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98)
             for at in (0.80, 0.84, 0.88, 0.92, 0.96)]

ml_A = {cfg: f05_for_mask(ml_mask(*cfg), half_A) for cfg in ML_GRID}
rule_A = {cfg: f05_for_mask(rule_mask(*cfg), half_A) for cfg in RULE_GRID}
best_ml_A, best_rule_A = max(ml_A, key=ml_A.get), max(rule_A, key=rule_A.get)

comparison = pd.DataFrame([
    {"matcher": "original hand-set rule (is_match)", "config": "-",
     "F0.5 tuning half A": f05_for_mask(_default_rule, half_A),
     "F0.5 held-out half B": f05_for_mask(_default_rule, half_B)},
    {"matcher": "rules, thresholds tuned on A", "config": f"name>={best_rule_A[0]}, addr>={best_rule_A[1]}",
     "F0.5 tuning half A": rule_A[best_rule_A],
     "F0.5 held-out half B": f05_for_mask(rule_mask(*best_rule_A), half_B)},
    {"matcher": "LightGBM, threshold tuned on A", "config": f"p>={best_ml_A[0]}, rel={best_ml_A[1]}",
     "F0.5 tuning half A": ml_A[best_ml_A],
     "F0.5 held-out half B": f05_for_mask(ml_mask(*best_ml_A), half_B)},
])
display(comparison.round(4))

USE_ML_MATCHER = comparison.loc[2, "F0.5 held-out half B"] > comparison.loc[1, "F0.5 held-out half B"]

# Final configuration: re-tune the winner's threshold on ALL validation entities.
_original_is_match = globals().get("_original_is_match", is_match)
rule_full = {cfg: f05_for_mask(rule_mask(*cfg), all_val) for cfg in RULE_GRID}
BEST_RULE_NT, BEST_RULE_AT = max(rule_full, key=rule_full.get)
if rule_full[(BEST_RULE_NT, BEST_RULE_AT)] >= f05_for_mask(_default_rule, all_val):
    is_match = make_rule(BEST_RULE_NT, BEST_RULE_AT)   # the final matching (Section 27) uses this
    print(f"Rule-based matching now uses the tuned thresholds: name>={BEST_RULE_NT}, addr>={BEST_RULE_AT}")
else:
    is_match = _original_is_match
    print("Original hand-set rule is still the best rule configuration -- kept.")

if USE_ML_MATCHER:
    ml_full = {cfg: f05_for_mask(ml_mask(*cfg), all_val) for cfg in ML_GRID}
    ML_THRESHOLD, ML_REL = max(ml_full, key=ml_full.get)
    print(f"LightGBM wins on held-out entities -> used for the test set "
          f"(p >= {ML_THRESHOLD}, rel = {ML_REL}; validation F0.5 {ml_full[(ML_THRESHOLD, ML_REL)]:.4f})")
else:
    print("Tuned rules win on held-out entities -> the rule-based matcher is used for the test set.")

## 24. Scalable inference design

The full test set is much larger than our validation sample.

For production inference we therefore use a **stream-friendly strategy**:

1. read the test sources
2. normalize only required fields
3. build compact indexes
4. process S1 in chunks
5. generate candidates by blocking
6. score only candidates
7. immediately write results

The key optimization is that fuzzy scoring happens only on blocked candidates — never on the full Cartesian product.

In [ ]:
# ============================================================
# 24. SCALABLE DATA LOADER
# ============================================================
def load_core(path, limit=None):
    kwargs = {
        "sep": "\t",
        "usecols": CORE_COLS,
        "dtype": str,
    }
    if limit is not None:
        kwargs["nrows"] = limit

    df = pd.read_csv(path, **kwargs).fillna("")
    return add_block_keys(prepare_small(df))

# Development mode:
# Upgrade: default is now the FULL test set. Dev mode only predicts the first 25k test S1, and the
# submission must contain every test S1 entity. Set True only for a quick smoke run.
# FAST_DEV_MODE = True  (original)
FAST_DEV_MODE = False

DEV_S1_LIMIT = 25_000
DEV_S2_LIMIT = 100_000
DEV_S3_LIMIT = 100_000

if FAST_DEV_MODE:
    test_s1 = load_core(FILES["test_s1"], DEV_S1_LIMIT)
    test_s2 = load_core(FILES["test_s2"], DEV_S2_LIMIT)
    test_s3 = load_core(FILES["test_s3"], DEV_S3_LIMIT)
else:
    test_s1 = load_core(FILES["test_s1"])
    # Upgrade: stream S2/S3, keeping only records that share a blocking key with some test S1.
    # Lossless for blocking; avoids holding every normalized S2/S3 row (plus a dict per row).
    _test_keysets = s1_key_sets([test_s1])
    test_s2 = load_core_pruned(FILES["test_s2"], _test_keysets)
    test_s3 = load_core_pruned(FILES["test_s3"], _test_keysets)
    # Original:
    # test_s2 = load_core(FILES["test_s2"])
    # test_s3 = load_core(FILES["test_s3"])

print("Test S1:", len(test_s1))
print("Test S2:", len(test_s2))
print("Test S3:", len(test_s3))

## 25. Build the final test indexes

We build separate indexes for Source 2 and Source 3.

Keeping them separate makes source provenance explicit and prevents accidental S1-to-S1 matches.

In [ ]:
# ============================================================
# 25. FINAL TEST INDEXES
# ============================================================
test_idx_s2 = make_indexes(test_s2)
test_idx_s3 = make_indexes(test_s3)

test_s2_by_id = test_s2.set_index("entity_id").to_dict("index")
test_s3_by_id = test_s3.set_index("entity_id").to_dict("index")

print("S2 name blocks:", len(test_idx_s2["name"]))
print("S3 name blocks:", len(test_idx_s3["name"]))

## 26. Generate final candidate pairs

This is the exact candidate layer that the challenge asks us to preserve in `candidate_pairs.tsv`.

For each S1:

```text
candidate_entity_ids = union(
    name block,
    address block,
    name+address block
)
```

The final matches must always be a subset of these candidates.

In [ ]:
# ============================================================
# 26. CANDIDATE GENERATION
# ============================================================
def candidate_ids_for_test_row(row):
    out = set()
    out.update(candidates_for_row(row, test_idx_s2))
    out.update(candidates_for_row(row, test_idx_s3))
    return sorted(out)

candidate_rows = []
t0 = time.time()

for row in test_s1.itertuples(index=False):
    cands = candidate_ids_for_test_row(row)
    candidate_rows.append({
        "source1_entity_id": row.entity_id,
        "candidate_entity_ids": ",".join(cands)
    })

candidate_df = pd.DataFrame(candidate_rows)

print("Candidate generation time:", round(time.time() - t0, 2), "seconds")
print("S1 entities:", len(candidate_df))
print("Average candidates:", candidate_df["candidate_entity_ids"].map(
    lambda x: len(x.split(",")) if x else 0
).mean())
display(candidate_df.head(10))

## 27. Score the final candidates

Now we apply the same conservative matching rule used during validation.

Important:

- We never create a match outside the candidate set.
- We preserve every Source 1 entity.
- We allow multiple matches.
- We allow an empty match list.
- We remove duplicates.

In [ ]:
# ============================================================
# 27. FINAL MATCHING
# ============================================================
def final_predictions_for_row(row):
    cands = candidate_ids_for_test_row(row)
    out = []

    # Convert row namedtuple to dict for pair_features compatibility
    row_dict = row._asdict() if hasattr(row, "_asdict") else row

    for cid in cands:
        if cid.startswith("S2-"):
            c = test_s2_by_id.get(cid)
        elif cid.startswith("S3-"):
            c = test_s3_by_id.get(cid)
        else:
            c = None

        if c is None:
            continue

        # Convert candidate namedtuple/object to dict if needed
        c_dict = c._asdict() if hasattr(c, "_asdict") else c

        f = pair_features(row_dict, c_dict)
        if is_match(f):
            out.append(cid)

    return sorted(set(out))

match_rows = []
t0 = time.time()

for row in test_s1.itertuples(index=False):
    matches = final_predictions_for_row(row)
    match_rows.append({
        "source1_entity_id": row.entity_id,
        "matched_entity_ids": ",".join(matches)
    })

matching_df = pd.DataFrame(match_rows)

print("Matching time:", round(time.time() - t0, 2), "seconds")
display(matching_df.head(10))

## 27b. Upgrade: apply the learned matcher to the test candidates

If LightGBM won the held-out comparison (23d), score every candidate pair in `candidate_df` in
batches and rebuild `matching_df`. The rule-based result above is kept as `matching_df_rules`.
Matches still come only from `candidate_pairs.tsv`, and every test S1 keeps exactly one row.

In [ ]:
# ============================================================
# 27b. LEARNED MATCHER ON THE TEST SET
# ============================================================
matching_df_rules = matching_df.copy()

if USE_ML_MATCHER:
    test_s1_lookup = lookup_frame([test_s1])
    test_cand_lookup = lookup_frame([test_s2, test_s3])
    CHUNK_S1 = 100_000
    ml_matches = defaultdict(list)
    t0 = time.time()
    for start in range(0, len(candidate_df), CHUNK_S1):
        chunk = candidate_df.iloc[start:start + CHUNK_S1]
        chunk = chunk[chunk["candidate_entity_ids"] != ""]
        if chunk.empty:
            continue
        pairs = (chunk.assign(candidate_entity_id=chunk["candidate_entity_ids"].str.split(","))
                      .explode("candidate_entity_id")[["source1_entity_id", "candidate_entity_id"]]
                      .reset_index(drop=True))
        p = predict_pairs(pair_feature_table(pairs, test_s1_lookup, test_cand_lookup))
        pmax = pd.Series(p).groupby(pairs["source1_entity_id"].to_numpy()).transform("max").to_numpy()
        keep = (p >= ML_THRESHOLD) & (p >= ML_REL * pmax)
        for s, c in zip(pairs["source1_entity_id"].to_numpy()[keep], pairs["candidate_entity_id"].to_numpy()[keep]):
            ml_matches[s].append(c)
        print(f"\rscored S1 {min(start + CHUNK_S1, len(candidate_df)):,}/{len(candidate_df):,} "
              f"({time.time() - t0:.0f}s)", end="")
    print()
    matching_df = pd.DataFrame({"source1_entity_id": test_s1["entity_id"].tolist()})
    matching_df["matched_entity_ids"] = [",".join(sorted(set(ml_matches.get(s, []))))
                                         for s in matching_df["source1_entity_id"]]
    _n = lambda d: int(d["matched_entity_ids"].map(lambda x: len(x.split(",")) if x else 0).sum())
    print(f"Predicted links -- rules: {_n(matching_df_rules):,} | LightGBM (used): {_n(matching_df):,}")
else:
    print("Rule-based matching kept (it won on held-out validation entities).")
display(matching_df.head(10))

## 28. Validate the output structure before saving

The challenge requires:

- exactly one row for every test Source 1 entity
- only S2/S3 IDs in match lists
- no duplicate IDs
- empty lists for predicted singletons
- matching IDs must also occur in the candidate file

The official challenge provides a validator for the full submission package; this notebook adds a lightweight in-notebook validation layer so structural errors are caught immediately.

In [ ]:
# ============================================================
# 28. NOTEBOOK VALIDATION
# ============================================================
def validate_outputs(s1_df, matching_df, candidate_df, s2_df, s3_df):
    problems = []

    expected = set(s1_df["entity_id"])
    predicted = set(matching_df["source1_entity_id"])

    if expected != predicted:
        problems.append(
            f"S1 coverage mismatch: expected {len(expected)}, got {len(predicted)}"
        )

    if matching_df["source1_entity_id"].duplicated().any():
        problems.append("Duplicate source1_entity_id in matching output.")

    valid_s2s3 = set(s2_df["entity_id"]) | set(s3_df["entity_id"])

    candidate_map = dict(zip(
        candidate_df["source1_entity_id"],
        candidate_df["candidate_entity_ids"].map(
            lambda x: set(x.split(",")) if x else set()
        )
    ))

    for row in matching_df.itertuples(index=False):
        mids = set(row.matched_entity_ids.split(",")) if row.matched_entity_ids else set()

        if len(mids) != len(row.matched_entity_ids.split(",")) if row.matched_entity_ids else False:
            problems.append(f"Duplicate IDs for {row.source1_entity_id}")

        bad_ids = mids - valid_s2s3
        if bad_ids:
            problems.append(f"Invalid IDs for {row.source1_entity_id}: {list(bad_ids)[:3]}")

        if not mids.issubset(candidate_map.get(row.source1_entity_id, set())):
            problems.append(f"Match outside candidate set: {row.source1_entity_id}")

    if problems:
        print("❌ Validation found issues:")
        for p in problems[:25]:
            print("-", p)
        return False

    print("✅ Notebook validation passed.")
    return True

validate_outputs(
    test_s1,
    matching_df,
    candidate_df,
    test_s2,
    test_s3
)

## 29. Write the required TSV outputs

The two files are written exactly with tab separators.

### `matching_results.tsv`

The leaderboard-scored prediction file.

### `candidate_pairs.tsv`

The final candidate set fed to the matching stage.

If you are developing with `FAST_DEV_MODE=True`, these outputs are only a **development sample**. Set it to `False` and rerun the inference section for the complete test set before submission.

In [ ]:
# ============================================================
# 29. SAVE OUTPUTS
# ============================================================
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matching_path = OUTPUT_DIR / "matching_results.tsv"
candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

matching_df.to_csv(matching_path, sep="\t", index=False)
candidate_df.to_csv(candidate_path, sep="\t", index=False)

print("Saved:")
print(matching_path)
print(candidate_path)

print("\nFile sizes:")
print("matching_results.tsv:", round(matching_path.stat().st_size / 1024**2, 2), "MB")
print("candidate_pairs.tsv:", round(candidate_path.stat().st_size / 1024**2, 2), "MB")

## 29b. Upgrade: run the official validator, if attached

Looks for the challenge's `validate_submission.py` anywhere under `/kaggle/input` or
`/kaggle/working`. Attach or upload it to get the official check; otherwise this cell just says
so, and the in-notebook validation in Section 28 still applies.

In [ ]:
# ============================================================
# 29b. OFFICIAL SUBMISSION VALIDATOR
# ============================================================
import subprocess
import sys

_validator = next((p for root in (Path("/kaggle/input"), Path("/kaggle/working")) if root.exists()
                   for p in sorted(root.rglob("validate_submission.py"))), None)
if _validator is None:
    print("validate_submission.py not found under /kaggle/input or /kaggle/working -- "
          "attach/upload it to run the official check.")
else:
    _cmd = [sys.executable, str(_validator), "--matching", str(matching_path),
            "--candidate", str(candidate_path), "--test-dir", str(FILES["test_s1"].parent)]
    print("Running:", " ".join(_cmd))
    _r = subprocess.run(_cmd, capture_output=True, text=True)
    print(_r.stdout[-5000:])
    if _r.returncode != 0:
        print(_r.stderr[-5000:])

## 30. Final prediction diagnostics

Before downloading anything, inspect:

- number of S1 entities
- predicted singleton rate
- total predicted links
- average matches per S1
- maximum matches
- candidate-set size

A surprisingly large number of matches can be a warning sign in a precision-heavy task; a surprisingly small number can indicate overly strict blocking.

In [ ]:
# ============================================================
# 30. FINAL DIAGNOSTICS
# ============================================================
match_counts = matching_df["matched_entity_ids"].map(
    lambda x: len(x.split(",")) if x else 0
)

candidate_counts = candidate_df["candidate_entity_ids"].map(
    lambda x: len(x.split(",")) if x else 0
)

diagnostics = pd.Series({
    "S1 entities": len(matching_df),
    "Predicted links": int(match_counts.sum()),
    "Predicted singleton S1": int((match_counts == 0).sum()),
    "Singleton rate %": round((match_counts == 0).mean() * 100, 2),
    "Average matches / S1": round(match_counts.mean(), 4),
    "Maximum matches / S1": int(match_counts.max()),
    "Average candidates / S1": round(candidate_counts.mean(), 4),
    "Maximum candidates / S1": int(candidate_counts.max()),
})

display(diagnostics.to_frame("value"))

## 31. Inspect the highest-match entities

Multiple matches are valid in this challenge, but unusually large match lists deserve inspection.

This table shows the entities with the largest predicted match sets so you can manually audit whether the blocking keys are too broad.

In [ ]:
# ============================================================
# 31. MULTI-MATCH AUDIT
# ============================================================
audit = matching_df.copy()
audit["match_count"] = match_counts

display(
    audit.sort_values("match_count", ascending=False)
         .head(30)
)

##  32. What this pipeline is doing well

### Strengths

- **Fast:** avoids all-pairs comparisons
- **Auditable:** every match has deterministic feature evidence
- **Precision-aware:** conservative decision rules
- **Open-set country handling:** France is not hard-coded
- **Multi-match aware:** one S1 can receive multiple S2/S3 matches
- **Submission-safe:** matches are restricted to generated candidates
- **Lightweight:** no large language model or external API is required

### Important limitation

This is a strong, interpretable baseline rather than a claim of optimal leaderboard performance.

For further improvement, the natural next step is to train a pairwise classifier on blocked candidate pairs using features such as:

- character n-gram similarity
- token-set similarity
- address-number agreement
- postal-code agreement
- token overlap
- country equality
- exact/partial field indicators

## 33. Optional upgrade path

If you want to push this notebook beyond the deterministic baseline, add the following in stages:

**Stage 1 — Better blocking**
- character prefixes
- rare-token inverted indexes
- address-number blocks
- postal-code blocks
- multiple blocking passes

**Stage 2 — Pairwise ML**
- generate positive pairs from ground truth
- generate hard negatives from the same blocks
- train LightGBM / XGBoost / Logistic Regression
- calibrate probabilities

**Stage 3 — Precision control**
- optimize threshold directly for macro F₀.₅
- add singleton confidence rules
- analyze false-merge clusters

**Stage 4 — Scale**
- Polars lazy execution
- chunked S1 inference
- compact integer IDs
- Parquet intermediates
- parallel candidate scoring

The challenge documentation emphasizes that blocking determines the recall ceiling, while the final matching model decides which candidates become actual links.

## 34. Challenge-aware checklist

Before submitting:

- [ ] Read all TSV files with `sep="\t"`
- [ ] Do not hard-code countries to only US/India
- [ ] Include every test S1 entity
- [ ] Allow empty match lists
- [ ] Allow multiple S2/S3 matches
- [ ] Never output S1 IDs as matches
- [ ] Remove duplicate IDs
- [ ] Ensure every final match exists in `candidate_pairs.tsv`
- [ ] Run the official `validate_submission.py`
- [ ] Package code + outputs + methodology as required

The official challenge description states that `matching_results.tsv` is the scored file and `candidate_pairs.tsv` is used to audit the candidate-generation stage.

In [ ]:
# ============================================================
# 35. FINAL QUICK CHECK
# ============================================================
print("=" * 70)
print("AMAZON ML CHALLENGE 2026 — FINAL CHECK")
print("=" * 70)
print("matching_results.tsv :", matching_path.exists())
print("candidate_pairs.tsv :", candidate_path.exists())
print("S1 rows              :", len(matching_df))
print("Predicted links      :", int(match_counts.sum()))
print("Predicted singletons :", int((match_counts == 0).sum()))
print("Avg candidates       :", round(candidate_counts.mean(), 3))
print("=" * 70)

# Helpful download links in the Kaggle output panel:
print(f"Output directory: {OUTPUT_DIR}")

# Final Takeaway — From Raw Records to Entity Links

You have now built a complete, publication-ready **Business Entity Resolution** workflow:

```text
RAW TSV FILES
     ↓
FAST SCHEMA CHECK
     ↓
SAMPLE-BASED EDA
     ↓
TEXT NORMALIZATION
     ↓
COUNTRY-AWARE BLOCKING
     ↓
CANDIDATE GENERATION
     ↓
PAIRWISE EVIDENCE SCORING
     ↓
PRECISION-ORIENTED MATCH DECISION
     ↓
F₀.₅ VALIDATION
     ↓
TEST INFERENCE
     ↓
candidate_pairs.tsv
     +
matching_results.tsv
```

### Key idea

The winning mindset for this type of problem is not:

> “Compare every business with every other business.”

It is:

> **“Find a small set of plausible candidates cheaply, then spend computation only where the evidence is meaningful.”**

That makes entity resolution scalable, explainable, and much more practical on a dataset containing millions of records.

### Official dataset context

The challenge defines Source 1 as the reference source and asks for all matching Source 2/Source 3 entities, with evaluation based on macro-averaged **F₀.₅**. The published dataset documentation also highlights the unseen-France test distribution and the importance of blocking.

**Notebook status:** ✅ EDA → ✅ preprocessing → ✅ blocking → ✅ matching → ✅ validation → ✅ submission files

> **Next upgrade:** replace the deterministic matcher with a trained pairwise ML model while keeping the same blocking layer. That is the natural route to a stronger competition solution without sacrificing the notebook's speed and interpretability.